### Libraries and HTTP Headers

The project relies on three core Python libraries to perform web scraping and organize the collected data. The requests library is responsible for sending HTTP requests to the Transfermarkt website, while BeautifulSoup (from the bs4 package) parses the HTML content and enables the extraction of the desired information. Finally, pandas is used to structure the extracted data into tabular formats, allowing it to be easily analyzed, manipulated, or exported to CSV files.

In [5]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import scraping_functions as sf
import importlib

importlib.reload(sf);

Additionally, a custom HTTP header containing a User-Agent string is included in every request. This header identifies the requests as originating from a standard web browser, reducing the likelihood of access restrictions and ensuring that the website returns the same content served to regular users. Using consistent headers also contributes to more reliable and stable data collection throughout the execution of the scraping functions.

In [2]:
headers = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0"
}

### Leagues Dictionary

To improve the flexibility and reusability of the scraping functions, a dictionary named all_leagues was created to centralize the mapping between each competition and its corresponding Transfermarkt competition code. Instead of hardcoding these identifiers throughout the project, the dictionary allows the target league to be selected dynamically, making the functions easier to maintain and extend.

This approach enables the same scraping logic to be reused across multiple competitions by simply referencing the desired league from the dictionary. As long as the target competition follows the same page structure on Transfermarkt, new leagues can be incorporated by adding a single key-value pair, without requiring any modifications to the scraping functions themselves.

In [3]:
all_leagues = {
	'premier-league' : 'GB1',
	'bundesliga' : 'L1',
	'serie-a' : 'IT1',
	'laliga' : 'ES1',
	'ligue-1' : 'FR1',
	'campeonato-brasileiro-serie-a' : 'BRA1'
}

## Functions

This project required the development of six specialized scraping functions responsible for collecting match results, match events, league standings, historical standings progression, titles, and squad information. The architecture was designed to be modular and reusable, enabling data extraction from both completed and ongoing seasons through round-specific queries. Furthermore, the same functions can be easily adapted to other competitions available on Transfermarkt, as long as they share the same underlying page structure. Each function and its role within the project are presented in the following sections.

### 1) get_events()

The get_events() function extracts all match events from a specific round of a Transfermarkt competition. It identifies the team involved, the minute of the event, the player responsible, and classifies the event into predefined categories, including regular goals, penalty goals, own goals, missed penalties, and red cards. The function also generates unique identifiers for each event and accounts for different page layouts to ensure accurate data extraction. The resulting data is returned in a structured format, ready for analysis or conversion into a DataFrame.

In [5]:
premier_events = []

for season in range(2025,2026):
    if season > 1994: season_round = 39
    else: season_round = 43

    for round in range(1,season_round+1):
        event_data = sf.get_events(headers,'premier-league',season,round)
        premier_events.extend(event_data[1:])

df_events = pd.DataFrame(premier_events, columns=event_data[0])
display(df_events)

,season_id,match_id,event_id,event_team,event_minute,event_type,event_player
0,GB1-2025,M-2025-01-01,E-2025-01-0001,Liverpool FC,37',1,Hugo Ekitiké
1,GB1-2025,M-2025-01-01,E-2025-01-0002,Liverpool FC,49',1,Cody Gakpo
2,GB1-2025,M-2025-01-01,E-2025-01-0003,AFC Bournemouth,64',1,Antoine Semenyo
3,GB1-2025,M-2025-01-01,E-2025-01-0004,AFC Bournemouth,76',1,Antoine Semenyo
4,GB1-2025,M-2025-01-01,E-2025-01-0005,Liverpool FC,88',1,Federico Chiesa
...,...,...,...,...,...,...,...
1099,GB1-2025,M-2025-38-08,E-2025-38-0021,Chelsea FC,62',-3,Wesley Fofana
1100,GB1-2025,M-2025-38-09,E-2025-38-0022,Tottenham Hotspur,43',1,João Palhinha
1101,GB1-2025,M-2025-38-10,E-2025-38-0023,West Ham United,67',1,Taty Castellanos
1102,GB1-2025,M-2025-38-10,E-2025-38-0024,West Ham United,79',1,Jarrod Bowen


### 2) get_match()

The get_match() function scrapes match-level information from a specific round of a Transfermarkt competition. It collects the participating teams, final score, match date, referee, attendance, and generates unique identifiers for the season, round, and match. The function also handles different page layouts caused by the presence of forum links, ensuring that team names are extracted correctly. Finally, the collected data is organized into a structured list, making it ready for further processing or conversion into a DataFrame.

In [8]:
premier_matches = []

for season in range(2000,2002):
    if season > 1994: season_round = 38
    else: season_round = 42

    for round in range(1,season_round+1):
        matches_data = sf.get_match(headers,'premier-league',season,round)
        premier_matches.extend(matches_data[1:])

df_premier_matches = pd.DataFrame(premier_matches, columns=matches_data[0])
display(df_premier_matches)

,season_id,round_id,match_id,home_team,final_score,away_team,date,referee,attendance
0,GB1-2000,R-2000-01,M-2000-01-001,Charlton Athletic,4:0,Manchester City,19/08/2000,Rob Harris,20.043
1,GB1-2000,R-2000-01,M-2000-01-002,Chelsea FC,4:2,West Ham United,19/08/2000,Graham Barber,34.914
2,GB1-2000,R-2000-01,M-2000-01-003,Coventry City,1:3,Middlesbrough FC,19/08/2000,Barry Knight,20.624
3,GB1-2000,R-2000-01,M-2000-01-004,Derby County,2:2,Southampton FC,19/08/2000,Andy D'Urso,27.223
4,GB1-2000,R-2000-01,M-2000-01-005,Leeds United,2:0,Everton FC,19/08/2000,Dermot Gallagher,40.010
...,...,...,...,...,...,...,...,...,...
755,GB1-2001,R-2001-38,M-2001-38-006,Leeds United,1:0,Middlesbrough FC,11/05/2002,Uriah Rennie,40.218
756,GB1-2001,R-2001-38,M-2001-38-007,Leicester City,2:1,Tottenham Hotspur,11/05/2002,David Elleray,21.716
757,GB1-2001,R-2001-38,M-2001-38-008,Sunderland AFC,1:1,Derby County,11/05/2002,Alan Wiley,47.989
758,GB1-2001,R-2001-38,M-2001-38-009,Manchester United,0:0,Charlton Athletic,11/05/2002,Graham Poll,67.571


### 3) get_placements()

The get_placements() function retrieves the league standings for a specific round of a Transfermarkt competition. It extracts each team's position, matches played, wins, draws, losses, goals scored, goal difference, and total points. Additionally, the function generates unique identifiers for the season and round, organizing the collected information into a structured dataset that can be easily analyzed or converted into a DataFrame.

In [10]:
premier_placements = []

for season in range(2024,2025):
    if season > 1994: season_round = 38
    else: season_round = 42

    for round in range(1,season_round+1):
        placements_data = sf.get_placements(headers,'premier-league', season, round)
        premier_placements.extend(placements_data[1:])

df_premier_placements = pd.DataFrame(premier_placements, columns=placements_data[0])
display(df_premier_placements)

,season_id,round_id,placement,team_name,matches,wins,draws,losses,goals,goal_dif,points
0,GB1-2024,R-2024-01,1,Brighton & Hove Albion,1,1,0,0,3:0,3,3
1,GB1-2024,R-2024-01,2,Arsenal FC,1,1,0,0,2:0,2,3
2,GB1-2024,R-2024-01,3,Liverpool FC,1,1,0,0,2:0,2,3
3,GB1-2024,R-2024-01,4,Manchester City,1,1,0,0,2:0,2,3
4,GB1-2024,R-2024-01,5,Aston Villa,1,1,0,0,2:1,1,3
...,...,...,...,...,...,...,...,...,...,...,...
755,GB1-2024,R-2024-38,16,Wolverhampton Wanderers,38,12,6,20,54:69,-15,42
756,GB1-2024,R-2024-38,17,Tottenham Hotspur,38,11,5,22,64:65,-1,38
757,GB1-2024,R-2024-38,18,Leicester City,38,6,7,25,33:80,-47,25
758,GB1-2024,R-2024-38,19,Ipswich Town,38,4,10,24,36:82,-46,22


### 4) get_squad()

The get_squad() function extracts squad-related information for every team participating in a Transfermarkt competition during a given season. It retrieves each team's market value, squad size, average player age, and number of foreign players. The function also handles slight variations in the page structure when extracting market values, ensuring consistent results. All collected information is organized into a structured dataset that can be easily analyzed or converted into a DataFrame.

In [13]:
premier_squad = []

# transfermarkt only contains values for team_value from 2004
for season in range(2020,2026):
    squad_data = sf.get_squad(headers,'premier-league',season)
    premier_squad.extend(squad_data[1:])

df_premier_squad = pd.DataFrame(premier_squad, columns=squad_data[0])
df_premier_squad.sort_values(['season_id','team_value'],ascending=[True,False],inplace=True,ignore_index=True)
display(df_premier_squad)

,season_id,team_name,team_value,team_squad,team_avg_age,team_foreigners
0,GB1-2020,Manchester City,1.040000e+09,36,25.3,23
1,GB1-2020,Liverpool FC,9.696500e+08,43,24.9,28
2,GB1-2020,Chelsea FC,8.892000e+08,39,25.7,23
3,GB1-2020,Manchester United,7.700500e+08,39,25.4,26
4,GB1-2020,Tottenham Hotspur,7.035000e+08,41,25.2,24
...,...,...,...,...,...,...
115,GB1-2025,Sunderland AFC,4.249300e+08,33,25.4,27
116,GB1-2025,Leeds United,3.733000e+08,28,27.4,21
117,GB1-2025,Fulham FC,3.562000e+08,25,28.1,21
118,GB1-2025,Wolverhampton Wanderers,3.183500e+08,30,26.2,25


### 5) get_title()

The get_title() function retrieves the historical champions of a Transfermarkt competition. For each title-winning season, it extracts the champion club and its manager, while also converting the season label into a standardized season identifier. In the case of the Premier League, the function considers only seasons from 1992 onward, when the competition adopted its current format. The collected data is returned as a structured dataset, ready for analysis or conversion into a DataFrame.

In [3]:
premier_titles = sf.get_title(headers,'premier-league')

df_premier_titles = pd.DataFrame(premier_titles[1:], columns=premier_titles[0])
display(df_premier_titles)

,season_id,season_name,team_name,manager_name
0,GB1-2025,25/26,Arsenal FC,Mikel Arteta
1,GB1-2024,24/25,Liverpool FC,Arne Slot
2,GB1-2023,23/24,Manchester City,Pep Guardiola
3,GB1-2022,22/23,Manchester City,Pep Guardiola
4,GB1-2021,21/22,Manchester City,Pep Guardiola
5,GB1-2020,20/21,Manchester City,Pep Guardiola
6,GB1-2019,19/20,Liverpool FC,Jürgen Klopp
7,GB1-2018,18/19,Manchester City,Pep Guardiola
8,GB1-2017,17/18,Manchester City,Pep Guardiola
9,GB1-2016,16/17,Chelsea FC,Antonio Conte


### 6) get_table()

The get_table() function retrieves the final league table for a specific season of a Transfermarkt competition. It extracts each team's final position, matches played, wins, draws, losses, goals scored, goal difference, and total points. The function generates a standardized season identifier and organizes the collected information into a structured dataset, making it suitable for analysis or conversion into a DataFrame.

In [4]:
premier_table = []

for season in range(2020,2026):
    table_data = sf.get_table(headers,'premier-league',season)
    premier_table.extend(table_data[1:])

df_premier_table = pd.DataFrame(premier_table,columns=table_data[0])
display(df_premier_table)

,season_id,pos,team_name,played,wins,draws,losses,goals,goal_dif,points
0,GB1-2020,1,Manchester City,38,27,5,6,83:32,51,86
1,GB1-2020,2,Manchester United,38,21,11,6,73:44,29,74
2,GB1-2020,3,Liverpool FC,38,20,9,9,68:42,26,69
3,GB1-2020,4,Chelsea FC,38,19,10,9,58:36,22,67
4,GB1-2020,5,Leicester City,38,20,6,12,68:50,18,66
...,...,...,...,...,...,...,...,...,...,...
115,GB1-2025,16,Nottingham Forest,38,11,11,16,48:51,-3,44
116,GB1-2025,17,Tottenham Hotspur,38,10,11,17,48:57,-9,41
117,GB1-2025,18,West Ham United,38,10,9,19,46:65,-19,39
118,GB1-2025,19,Burnley FC,38,4,10,24,38:75,-37,22


### 7) get_top_scorers

The get_top_scorers() function retrieves the top scorers for a specific season of a Transfermarkt competition. Since the ranking spans multiple pages, the function first determines the total number of pages and then iterates through each one to collect all player records. For every player, it extracts their ranking position, nationality, age, club, matches played, and goals scored. The function also accounts for players who represented multiple clubs during the season, ensuring consistent data extraction. The collected information is returned as a structured dataset, ready for analysis or conversion into a DataFrame.

In [7]:
premier_top_scorers = []

for season in range(2024,2026):
    table_data = sf.get_top_scorers(headers,'premier-league',season)
    premier_top_scorers.extend(table_data[1:])

df_premier_top_scorers = pd.DataFrame(premier_top_scorers,columns=table_data[0])
df_premier_top_scorers.sort_values(['season_id','pos'],inplace=True,ignore_index=True)
display(df_premier_top_scorers)

,season_id,pos,country,age,player_name,team,matches,goals
0,GB1-2024,1,Egypt,32,Mohamed Salah,Liverpool FC,38,29
1,GB1-2024,2,Sweden,25,Alexander Isak,Newcastle United,34,23
2,GB1-2024,3,Norway,24,Erling Haaland,Manchester City,31,22
3,GB1-2024,4,Cameroon,25,Bryan Mbeumo,Brentford FC,38,20
4,GB1-2024,5,New Zealand,33,Chris Wood,Nottingham Forest,36,20
...,...,...,...,...,...,...,...,...
546,GB1-2025,276,Denmark,25,Matt O'Riley,Brighton & Hove Albion,6,1
547,GB1-2025,277,Portugal,23,Fábio Carvalho,Brentford FC,6,1
548,GB1-2025,278,Italy,25,Lorenzo Lucca,Nottingham Forest,4,1
549,GB1-2025,279,Wales,32,Ben Davies,Tottenham Hotspur,3,1
